# 让系统选择资料来源

问题已经明确主题时，先缩小资料范围可能减少无关命中。本页用规则路由选择《南瓜书》的章节：先比较强化学习问题，再用原本能答对的规则学习问题复查。

## 原理与实验设置

查询路由（Query Routing）根据问题和资料目录选择一个或多个检索范围，再在所选范围内查找。规则路由适合主题词和边界稳定的资料；模型路由可处理灵活表达，但需另外检查误路由和调用成本。

`source_catalog` 预先登记主题词、页范围与标签，`choose_source` 只使用问题文字，检索结束后才读取预期页评分。两边均取前 3 页，本页不生成回答。跨文件实验见[构建多轮多来源助手](构建多轮多来源助手.ipynb)，模型选择检索工具见[让系统决定怎样检索](让系统决定怎样检索.ipynb)。


In [1]:
import sys
from pathlib import Path

def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / 'data' / 'dataset/manifest.json').is_file():
            return folder
    raise FileNotFoundError('没有找到教程数据目录，请从本节所在目录运行。')

course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import build_bm25_search, load_query_catalog, load_pdf_pages
from common.nontraining_utils import load_annotation

data = load_query_catalog()
cases = {item['id']: item for item in data}
pages = load_pdf_pages()
full_search = build_bm25_search(pages)

# 资料范围是在整理文档时登记的章节，不从问题集的 expected_pages 反推。
source_catalog = {
    'lda_derivation': {
        'terms': ('LDA', '线性判别分析', '广义特征值'),
        'pages': range(41, 45),
        'label': '线性判别分析推导（第 41～44 页）',
    },
    'rule_learning': {
        'terms': ('规则学习', '可解释性'),
        'pages': range(191, 193),
        'label': '规则学习（第 191～192 页）',
    },
    'reinforcement_learning': {
        'terms': ('强化学习', 'Bellman'),
        'pages': range(193, 196),
        'label': '强化学习（第 193～195 页）',
    },
}

def choose_source(question):
    scored = {
        name: sum(term in question for term in info['terms'])
        for name, info in source_catalog.items()
    }
    best = max(scored, key=scored.get)
    return best if scored[best] else None

def rank_and_coverage(results, expected_pages):
    expected = set(expected_pages)
    found = sorted(expected.intersection(item.page for item in results))
    rank = next((index for index, item in enumerate(results, 1) if item.page in expected), None)
    return rank, found

for label, case_id in (
    ('主要问题：先选强化学习资料范围', 'bellman_with_reinforcement_scope'),
    ('复查：先选规则学习资料范围', 'rule_learning_interpretability'),
):
    case = cases[case_id]
    route = choose_source(case['query'])
    source = source_catalog[route]
    routed_pages = [page for page in pages if page['page'] in source['pages']]
    routed_search = build_bm25_search(routed_pages)
    baseline = full_search(case['query'], top_k=3)
    routed = routed_search(case['query'], top_k=3)
    annotation = load_annotation(case['id'])
    baseline_rank, baseline_found = rank_and_coverage(baseline, annotation['expected_pages'])
    routed_rank, routed_found = rank_and_coverage(routed, annotation['expected_pages'])
    print('\n' + label)
    print('问题：', case['query'])
    print('选择来源：', source['label'])
    print('整本书前 3 页：', [item.page for item in baseline], '；必要页：', baseline_found, '；首个必要页排名：', baseline_rank or '未出现')
    print('路由后前 3 页：', [item.page for item in routed], '；必要页：', routed_found, '；首个必要页排名：', routed_rank or '未出现')
    print('路由后的第一条原文摘要：', routed[0].text[:150], '...')
    if case_id == 'bellman_with_reinforcement_scope':
        assert not baseline_found and routed_found == [194]
    else:
        assert baseline_rank == 1 and routed_rank == 1


主要问题：先选强化学习资料范围
问题： 强化学习里当前和未来怎么联系？
选择来源： 强化学习（第 193～195 页）
整本书前 3 页： [2, 104, 59] ；必要页： [] ；首个必要页排名： 未出现
路由后前 3 页： [193, 194, 195] ；必要页： [194] ；首个必要页排名： 2
路由后的第一条原文摘要： 第16 章 强化学习 强化学习作为机器学习的子领域，其本身拥有一套完整的理论体系，以及诸多经典和最新前沿算 法，“西瓜书”该章内容仅可作为综述查阅，若想深究建议查阅其他相关书籍（例如《Easy RL：强化 学习教程》 [1]）进行系统性学习。 16.1 任务与奖赏 本节理解强化学习的定义和相关术语的 ...

复查：先选规则学习资料范围
问题： 规则学习为什么具有良好的可解释性？
选择来源： 规则学习（第 191～192 页）
整本书前 3 页： [191, 151, 14] ；必要页： [191] ；首个必要页排名： 1
路由后前 3 页： [191, 192] ；必要页： [191] ；首个必要页排名： 1
路由后的第一条原文摘要： 第15 章 规则学习 规则学习是“符号主义学习”的代表性方法，用来从训练数据中学到一组能对未见示例进行判别的规 则，形如“如果A 或B，并且C 的条件下，D 满足”这样的形式。因为这种学习方法更加贴合人类从数 据中学到经验的描述，具有非常良好的可解释性，是最早开始研究机器学习的技术之一。 15.1  ...


## 结果与限制

强化学习问题直接检索整本书时，前 3 页没有第 194 页；选择强化学习章节后，第 194 页进入前 3 条。规则学习复查题在两种设置下都把第 191 页排在第 1，原本命中的结果保持不变。

路由选错时，后续检索看不到正确证据。因此要记录所选范围与命中；来源不确定时可以扩大到多个范围或请求补充。下面保存两题的排名与页面覆盖，结论限于单份 PDF 的章节路由。


## 扩展到多个真实文件

Multi-Document Agent 为每个来源建立独立检索器，经路由后分别检索、合并证据，保留来源身份、命中记录与引用。实际接入还要分别处理访问范围和版本。

本课程的[构建多轮多来源助手](构建多轮多来源助手.ipynb)已直接读取三份当前文件：`data/dataset/manifest.json`、`2. 数据处理/README.md`、`7. 评估/README.md`，演示各源检索、追问、引用绑定与资料不足处理。它与本页的章节范围实验分别保存结果。


In [2]:
from common.eval_utils import emit_tutorial_audit

# 统一保存契约：路由和检索完成后才读取 expected_pages。
import json

def _actual_pages(items):
    pages = []
    for item in items:
        page = int(item.page)
        if page not in pages:
            pages.append(page)
    return pages

def _metrics(items, expected_pages):
    pages = _actual_pages(items)
    expected = {int(page) for page in expected_pages}
    found = set(pages) & expected
    rank = next((index for index, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': rank,
            'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def _route_result(case_id):
    case = cases[case_id]
    source = source_catalog[choose_source(case['query'])]
    routed_search = build_bm25_search([page for page in pages if page['page'] in source['pages']])
    baseline = full_search(case['query'], top_k=3)
    routed = routed_search(case['query'], top_k=3)
    # 两条检索已经完成，下面才读取标注并计算排名/覆盖率。
    annotation = load_annotation(case_id)
    expected = annotation['expected_pages']
    return baseline, routed, expected

def _emit(role, case_id, before_items, after_items, purpose=None):
    expected = _route_result(case_id)[2]
    payload = {'case_id': case_id, 'method': '按关键词选择资料来源（Query Routing）', 'role': role,
              'before': _metrics(before_items, expected),
              'after': _metrics(after_items, expected)}
    if purpose:
        payload['check_purpose'] = purpose
    emit_tutorial_audit(payload)

main_baseline, main_routed, _ = _route_result('bellman_with_reinforcement_scope')
check_baseline, check_routed, _ = _route_result('rule_learning_interpretability')
_emit('main', 'bellman_with_reinforcement_scope', main_baseline, main_routed)
_emit('check', 'rule_learning_interpretability', check_baseline, check_routed, '确认没有改坏')


## 从资料路由继续到 GraphRAG

Query Routing 回答“应该去哪个资料范围查”，GraphRAG 回答“实体之间通过哪些关系相连”。两者都可能缩小检索范围，但索引结构不同：本页前面的路由器仍检索原文片段；GraphRAG 会先从资料中构建带来源的 `(subject, relation, object)` 三元组或社区结构，再按问题中的实体遍历相关边，最后把命中的边映回原文证据。

一个可核查的 GraphRAG 流程至少包含：

1. 从原文抽取实体和关系，并为每条边保存 `evidence_id/source/page/quote`；结构化输出不合规时直接拒绝入图。
2. 对同名实体做消歧和合并，保留别名、版本和来源，不把字符串相同直接当成同一对象。
3. 从 query 提取种子实体，在限制跳数、边类型和访问范围后遍历子图。
4. 将关系路径映回连续原文，回答时引用原始 evidence，而不是把图中边本身当成最终事实。
5. 用多跳覆盖率、路径正确性、证据忠实度、延迟和建图成本评估；必须与同语料的向量或混合检索比较。

```python
# 教学接口：edge 必须绑定已经核验的原文证据。
edge = {
    'subject': '交叉验证法',
    'relation': '属于',
    'object': '模型评估方法',
    'evidence_id': '...',
    'source': 'pumpkin_book.pdf',
    'page': 18,
    'quote': '...',
}
# retrieve_subgraph(seed_entities, allowed_relations, max_hops=2)
# -> paths + bound evidence；随后仍要用原文 quote 组成回答上下文。
```

GraphRAG 更适合跨片段关系、多跳路径和全局结构问题；单页定义、精确术语或普通语义匹配通常不需要先建图。下面用两条来自 canonical evidence 的教学手工标注边做一个可执行 fixture：关系字段不是自动抽取结果，也没有经过独立人工语义复核；每条边仍必须回到南瓜书 PDF 的逐字 quote。

### 可核查的最小 GraphRAG fixture

本例使用 canonical 问题 `lda_multihop`：问题先给出“监督降维”的线索，再追问 N−1 个最大广义特征值及其特征向量。图只覆盖第 41、44 页，并把同一概念的 `LDA` 与 `线性判别分析` 合并为一个有别名的节点。边的主语、关系和宾语是教程作者依据上下文给出的教学标注，未做独立人工语义核验；`evidence_id`、`source`、`page`、`quote` 则从 canonical evidence 读取，并再次对照规范化 PDF 的 offsets。

遍历使用双向邻接，但把原始边方向保留在路径中；`max_hops=2` 和允许关系集合都是显式限制。对照基线是在完全相同的第 41、44 页范围内运行 BM25，故这里只展示一个小 fixture 的路径与证据命中边界，不声称自动抽取质量或全库收益。

In [3]:
from collections import defaultdict, deque

from common.dataset import (
    CANONICAL_DOCUMENT_ID,
    CANONICAL_PDF_RELATIVE_PATH,
    NORMALIZATION_VERSION,
    load_evidence_records,
)
from common.eval_utils import normalize_text

# 关系三元组是本 Notebook 的小型、确定性教学手工标注 fixture；
# quote 和页码不手写，统一从 canonical evidence 绑定。
GRAPH_EDGE_SPECS = (
    {
        'subject': 'entity:lda',
        'relation': 'is_a',
        'object': 'entity:supervised_reduction',
        'evidence_id': 'evi_677d8872b6b8',
    },
    {
        'subject': 'entity:lda',
        'relation': 'selects_under_constraint',
        'object': 'entity:selected_eigenvectors',
        'evidence_id': 'evi_94ff9315c02b',
    },
)

GRAPH_NODES = {
    'entity:lda': {
        'canonical_name': '线性判别分析（LDA）',
        'aliases': ('LDA', '线性判别分析'),
        'entity_type': 'method',
        'source': CANONICAL_PDF_RELATIVE_PATH,
        'version': NORMALIZATION_VERSION,
        'pages': (41, 44),
    },
    'entity:supervised_reduction': {
        'canonical_name': '监督降维',
        'aliases': ('监督降维',),
        'entity_type': 'method_family',
        'source': CANONICAL_PDF_RELATIVE_PATH,
        'version': NORMALIZATION_VERSION,
        'pages': (41,),
    },
    'entity:selected_eigenvectors': {
        'canonical_name': 'N−1 个最大广义特征值对应的特征向量',
        'aliases': ('最大广义特征值对应的特征向量',),
        'entity_type': 'result',
        'source': CANONICAL_PDF_RELATIVE_PATH,
        'version': NORMALIZATION_VERSION,
        'pages': (44,),
    },
}

# 只允许已登记的别名进入图；同名文本不会自动创建新节点。
ENTITY_ALIASES = {
    normalize_text(alias).casefold(): node_id
    for node_id, node in GRAPH_NODES.items()
    for alias in node['aliases']
}
SEED_ALIASES = {
    normalize_text(alias).casefold(): node_id
    for alias, node_id in (
        ('LDA', 'entity:lda'),
        ('线性判别分析', 'entity:lda'),
        ('监督降维', 'entity:supervised_reduction'),
    )
}

canonical_evidence = load_evidence_records()
evidence_by_id = {row['evidence_id']: row for row in canonical_evidence}
page_text_by_number = {int(row['page']): normalize_text(row['text']) for row in pages}

def resolve_entity(mention):
    key = normalize_text(mention).casefold()
    try:
        return ENTITY_ALIASES[key]
    except KeyError as error:
        raise ValueError(f'未登记的实体别名：{mention!r}') from error


def extract_seed_entities(query):
    normalized_query = normalize_text(query).casefold()
    matches = []
    for alias in sorted(SEED_ALIASES, key=len, reverse=True):
        if alias in normalized_query:
            node_id = SEED_ALIASES[alias]
            if node_id not in matches:
                matches.append(node_id)
    if not matches:
        raise ValueError('查询没有命中已登记的 GraphRAG 种子实体')
    return matches


def bind_edge(spec):
    evidence = evidence_by_id.get(spec['evidence_id'])
    if evidence is None:
        raise KeyError(f'canonical evidence 不存在：{spec["evidence_id"]}')
    if evidence.get('doc_id') != CANONICAL_DOCUMENT_ID:
        raise ValueError('GraphRAG fixture 只能绑定 canonical 南瓜书文档')
    for endpoint in (spec['subject'], spec['object']):
        if endpoint not in GRAPH_NODES:
            raise ValueError(f'边引用了未登记节点：{endpoint}')
    page = int(evidence['page'])
    quote = str(evidence['quote'])
    page_text = page_text_by_number.get(page)
    if page_text is None:
        raise ValueError(f'证据页不在已加载 PDF 中：{page}')
    offsets = evidence.get('offsets', {})
    start, end = offsets.get('start'), offsets.get('end')
    if page_text[start:end] != quote or end - start != len(quote):
        raise ValueError(f'证据 quote 未通过 PDF offsets 核验：{spec["evidence_id"]}')
    return {
        **spec,
        'source': CANONICAL_PDF_RELATIVE_PATH,
        'page': page,
        'quote': quote,
    }


GRAPH_EDGES = [bind_edge(spec) for spec in GRAPH_EDGE_SPECS]

def retrieve_subgraph(seed_entities, allowed_relations, max_hops=2):
    if not isinstance(max_hops, int) or isinstance(max_hops, bool) or max_hops not in (1, 2):
        raise ValueError('教学 fixture 只允许 1 或 2 跳')
    allowed = set(allowed_relations)
    if not allowed:
        raise ValueError('allowed_relations 不能为空')
    seed_ids = [
        item if item in GRAPH_NODES else resolve_entity(item)
        for item in seed_entities
    ]
    adjacency = defaultdict(list)
    for edge in GRAPH_EDGES:
        if edge['relation'] not in allowed:
            continue
        # 图遍历允许反向发现，但 path 保留原始边方向和证据。
        adjacency[edge['subject']].append((edge['object'], edge, 'forward'))
        adjacency[edge['object']].append((edge['subject'], edge, 'reverse'))
    queue = deque((seed, [seed], []) for seed in seed_ids)
    paths = []
    while queue:
        node_id, node_path, edge_path = queue.popleft()
        if edge_path:
            paths.append({
                'node_ids': node_path,
                'nodes': [GRAPH_NODES[item]['canonical_name'] for item in node_path],
                'edges': edge_path,
                'hops': len(edge_path),
            })
        if len(edge_path) >= max_hops:
            continue
        for next_node, edge, direction in adjacency[node_id]:
            if next_node in node_path:
                continue
            queue.append((next_node, node_path + [next_node], edge_path + [{
                'edge': edge,
                'direction': direction,
            }]))
    return paths

assert len(GRAPH_EDGES) == 2
assert all({
    'subject', 'relation', 'object', 'evidence_id', 'source', 'page', 'quote'
}.issubset(edge) for edge in GRAPH_EDGES)
assert resolve_entity('LDA') == resolve_entity('线性判别分析') == 'entity:lda'


### 运行路径，并与同范围 BM25 对照

基线只取同一证据范围的 top-1 页面；GraphRAG 从问题中的“监督降维”种子开始，最多走两跳，并把每条边映回原文 quote。评价字段在两条检索完成后才读取。

In [4]:
from common.dataset import load_qrel_records

graph_case_id = 'lda_multihop'
graph_query = cases[graph_case_id]['query']
scope_pages = {41, 44}
scoped_pages = [row for row in pages if row['page'] in scope_pages]
scoped_bm25 = build_bm25_search(scoped_pages)

# 先完成两种检索；此处还没有读取 expected_pages 或 qrels。
baseline_items = scoped_bm25(graph_query, top_k=1)
seed_entities = extract_seed_entities(graph_query)
allowed_relations = {'is_a', 'selects_under_constraint'}
all_paths = retrieve_subgraph(seed_entities, allowed_relations, max_hops=2)
target_paths = [
    path for path in all_paths
    if path['node_ids'][-1] == 'entity:selected_eigenvectors'
]
if not target_paths:
    raise AssertionError('两跳 GraphRAG 路径没有到达目标节点')

# 检索结束后再读取 canonical qrels，得到该真实多跳题的必要 evidence。
gold_evidence_ids = [
    row['evidence_id']
    for row in load_qrel_records()
    if row['query_id'] == graph_case_id and row['essential'] and row['relevance'] == 1
]
gold_evidence_set = set(gold_evidence_ids)

def evidence_ids_in_page_results(items):
    result = []
    for evidence in canonical_evidence:
        if any(
            int(item.page) == int(evidence['page']) and evidence['quote'] in item.text
            for item in items
        ):
            result.append(evidence['evidence_id'])
    return result

baseline_evidence_ids = evidence_ids_in_page_results(baseline_items)
baseline_hits = [item for item in baseline_evidence_ids if item in gold_evidence_set]
graph_evidence_ids = []
for path in target_paths:
    for hop in path['edges']:
        evidence_id = hop['edge']['evidence_id']
        if evidence_id not in graph_evidence_ids:
            graph_evidence_ids.append(evidence_id)
graph_hits = [item for item in graph_evidence_ids if item in gold_evidence_set]

def _coverage(hits):
    return len(hits) / len(gold_evidence_set) if gold_evidence_set else 0.0

path_report = [
    {
        'nodes': path['nodes'],
        'hops': path['hops'],
        'directions': [hop['direction'] for hop in path['edges']],
        'evidence_ids': [hop['edge']['evidence_id'] for hop in path['edges']],
        'quotes': [hop['edge']['quote'] for hop in path['edges']],
    }
    for path in target_paths
]
graph_audit = {
    'case_id': graph_case_id,
    'method': '受限 GraphRAG（确定性南瓜书 fixture）',
    'role': 'experiment',
    'query': graph_query,
    'fixture': {
        'automatic_extraction': False,
        'model_called': False,
        'source': CANONICAL_PDF_RELATIVE_PATH,
        'scope_pages': sorted(scope_pages),
    },
    'edges': GRAPH_EDGES,
    'nodes': GRAPH_NODES,
    'entity_resolution': {
        'canonical_node': 'entity:lda',
        'aliases': ['LDA', '线性判别分析'],
        'source': CANONICAL_PDF_RELATIVE_PATH,
        'version': NORMALIZATION_VERSION,
    },
    'seed_entities': [GRAPH_NODES[item]['canonical_name'] for item in seed_entities],
    'allowed_relations': sorted(allowed_relations),
    'max_hops': 2,
    'paths': path_report,
    'gold': {
        'essential_evidence_ids': gold_evidence_ids,
        'required_pages': [41, 44],
    },
    'baseline': {
        'method': 'BM25 page retrieval',
        'scope_pages': sorted(scope_pages),
        'top_k': 1,
        'retrieved_pages': [int(item.page) for item in baseline_items],
        'evidence_ids': baseline_evidence_ids,
        'required_evidence_hits': baseline_hits,
        'required_evidence_coverage': _coverage(baseline_hits),
    },
    'graph': {
        'retrieved_evidence_ids': graph_evidence_ids,
        'required_evidence_hits': graph_hits,
        'required_evidence_coverage': _coverage(graph_hits),
    },
    'limitations': [
        '关系三元组是依据本页两个证据片段给出的教学手工标注，未经独立人工语义核验，也未测试自动实体/关系抽取。',
        '仅覆盖一个 canonical 多跳问题和第 41、44 页，不代表全库或普遍提升。',
        'BM25 top-1 与两跳图遍历的预算不同；结果只说明本 fixture 的证据边界。',
    ],
}

print('问题：', graph_query)
print('种子实体：', graph_audit['seed_entities'])
print('允许关系：', graph_audit['allowed_relations'], '；最大跳数：', graph_audit['max_hops'])
print('目标路径：')
for path in path_report:
    print('  ', ' -> '.join(path['nodes']), '；evidence：', path['evidence_ids'])
print('BM25 同范围 top-1 页面：', graph_audit['baseline']['retrieved_pages'],
      '；必要 evidence 命中：', baseline_hits,
      '；覆盖率：', graph_audit['baseline']['required_evidence_coverage'])
print('GraphRAG 路径必要 evidence 命中：', graph_hits,
      '；覆盖率：', graph_audit['graph']['required_evidence_coverage'])
print('边回映原文摘要：')
for edge in GRAPH_EDGES:
    print(f"  {edge['subject']} --{edge['relation']}--> {edge['object']}")
    print('    ', edge['source'], 'p.', edge['page'], edge['quote'][:100], '...')

assert all(path['hops'] <= 2 for path in path_report)
assert set(graph_hits) == gold_evidence_set
assert len(baseline_hits) < len(graph_hits)
emit_tutorial_audit(graph_audit)


问题： 哪种监督降维方法能让同类样本投影接近、异类样本投影疏远？进一步说明其 N−1 个最大广义特征值与特征向量结论。
种子实体： ['监督降维']
允许关系： ['is_a', 'selects_under_constraint'] ；最大跳数： 2
目标路径：
   监督降维 -> 线性判别分析（LDA） -> N−1 个最大广义特征值对应的特征向量 ；evidence： ['evi_677d8872b6b8', 'evi_94ff9315c02b']
BM25 同范围 top-1 页面： [41] ；必要 evidence 命中： ['evi_677d8872b6b8'] ；覆盖率： 0.5
GraphRAG 路径必要 evidence 命中： ['evi_677d8872b6b8', 'evi_94ff9315c02b'] ；覆盖率： 1.0
边回映原文摘要：
  entity:lda --is_a--> entity:supervised_reduction
     data/pumpkin_book.pdf p. 41 由向量内积的几何意义可知，y 可以看作是x 在w 上的投影，因此在训练集上学得的模型能够保证训练 集中的同类样本在w 上的投影y 很相近，而异类样本在w 上的投影y 很疏远。然后对于新的测试样本 xi ...
  entity:lda --selects_under_constraint--> entity:selected_eigenvectors
     data/pumpkin_book.pdf p. 44 由于存在约束tr(WTSwW) = N−1 P i=1 wT i Swwi = 1，所以欲使上式取到最大值，只需取N −1 个最大的λi 即 可。根据Sbwi = λiSwwi 可知，λi 对应的便是 ...


## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[串起多来源会话](构建多轮多来源助手.ipynb)

